# 第 8 课 · CSS 选择器（soup.select）

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 第 7 课用 `find / find_all` 能找，但写起来啰嗦。这一课学 **CSS 选择器**：
> 一句话表达「我要哪些节点」，前端工程师怎么写，我们就怎么抄。本课只引入 `select`、`.`、`#`、空格、`[属性]`。

In [1]:
from pathlib import Path
from bs4 import BeautifulSoup
import bs4
print('bs4 版本：', bs4.__version__)

DATA_DIR = Path(r'C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\data')
books_html = (DATA_DIR / 'books_index.html').read_text(encoding='utf-8')
quotes_html = (DATA_DIR / 'quotes.html').read_text(encoding='utf-8')
books_soup = BeautifulSoup(books_html, 'lxml')
quotes_soup = BeautifulSoup(quotes_html, 'lxml')

bs4 版本： 4.15.0


## 8.1 类选择器 .product_pod：按 class 圈一排

**生活化类比**：图书馆门口的分类导览牌——你指着「文学区」三个字，整个书架上贴了这张牌的书全被圈出来。

**是什么**：`soup.select('.product_pod')` 返回**所有**带 `class="product_pod"` 的节点列表（注意前面那个点 `.`）。
`soup.select_one('.price_color')` 只要第一个。

**为什么需要**：和 `find_all(class_=...)` 等价，但写法更短、更通用。

In [2]:
pods = books_soup.select('.product_pod')
print('选中了几个书盒：', len(pods))
print('第一个书盒的价格：', pods[0].select_one('.price_color').get_text())

选中了几个书盒： 4
第一个书盒的价格： ¥51.77


## 8.2 id 选择器 #item-list：按“全楼唯一的门牌”找

**生活化类比**：class 是「文学区」这种可重复的分类牌；**id 是门牌号码**，全页面唯一。

**是什么**：`soup.select('#item-list')` 找 `id="item-list"` 的那个节点（注意前面是 `#`）。

**为什么需要**：有些元素（尤其动态生成的列表）只给 id 不给 class，这时候只能靠 `#`。

In [3]:
dynamic_html = (DATA_DIR / 'dynamic_demo.html').read_text(encoding='utf-8')
dyn_soup = BeautifulSoup(dynamic_html, 'lxml')
ul = dyn_soup.select_one('#item-list')
print('找到的节点：', ul)
print('它里面现在有几个 li：', len(ul.find_all('li')))
# 预期 0 个：那些 li 是 JavaScript 运行后才塞进去的，原始 HTML 里是空的——第 11 课解决

找到的节点： <ul id="item-list">
<!-- 预期被 JS 注入：
        <li class="item">机械键盘 · ¥299</li>
        <li class="item">无线鼠标 · ¥89</li>
        <li class="item">显示器支架 · ¥159</li>
        -->
</ul>
它里面现在有几个 li： 0


## 8.3 后代组合（空格）：逐级下钻

**生活化类比**：「图书馆 › 三楼 › 文学区 › 第三排」，用空格把每一级连起来。

**是什么**：`'section.product_list h3 a'` 的意思是——在 `section.product_list` **内部任意深处**，找 `h3` 里的 `a`。
空格 =「里面任意层次」。

**为什么需要**：全网有几十个 a，你只想拿「书盒里的书名链接」，逐级限定才不会抓错。

In [4]:
links = books_soup.select('section.product_list h3 a')
for a in links:
    print(a.get('title'))

A Light in the Attic
Tipping the Velvet
Soumission
Sharp Objects


## 8.4 属性选择器 a[title]：只要“带某个属性”的

**是什么**：`select('a[title]')` 选中所有**带 `title` 属性**的 `<a>`；也能写 `a[href="books_page2.html"]` 精确匹配。

**为什么需要**：有时节点没有 class，但有某个标志性属性（如 title、rel、href），就靠它。

In [5]:
for a in books_soup.select('a[title]'):
    print(a.get('title'), '->', a.get('href'))

A Light in the Attic -> catalogue/a-light-in-the-attic_1000/index.html
Tipping the Velvet -> catalogue/tipping-the-velvet_999/index.html
Soumission -> catalogue/soumission_998/index.html
Sharp Objects -> catalogue/sharp-objects_997/index.html


## 8.5 实战：用 quotes.html 把名言一次抠全

对照 quotes.html 的结构：每个名言是一个 `div.quote`，里面有 `span.text`（名言）、`small.author`（作者）、
`div.tags > a.tag`（标签）。用 8.1～8.4 学到的写法组合起来。

In [6]:
for q in quotes_soup.select('div.quote'):
    text = q.select_one('span.text').get_text()
    author = q.select_one('small.author').get_text()
    tags = [a.get_text() for a in q.select('a.tag')]
    print(author, '|', text)
    print('    标签：', tags)

—— 温斯顿·格鲁姆 | “生活就像一盒巧克力，你永远不知道下一颗会是什么味道。”
    标签： ['life', 'attributed']
—— 孔子 | “知之者不如好之者，好之者不如乐之者。”
    标签： ['education', 'learning']
—— Steve Jobs | “Stay hungry, stay foolish.”
    标签： ['learning', 'inspirational']


## 8.6 和 find_all 对照（两种写法都行，挑顺手的）

| 想找什么 | CSS 选择器写法 | 等价的 find 写法 |
|---|---|---|
| 所有书盒 | `.product_pod` | `find_all('article', class_='product_pod')` |
| 第一个书盒 | `.product_pod`[0] | `find('article', class_='product_pod')` |
| 书盒里的书名链接 | `section.product_list h3 a` | 层层 `find` |
| 有名言的小块 | `div.quote` | `find_all('div', class_='quote')` |

## 8.7 随堂小练习（自己动手，空格里写）

In [7]:
# TODO：用 select 取出 quotes.html 里全部作者，放进一个列表并打印

In [8]:
# TODO：用后代选择器写出“选中 nav.pagination 里那个 a”，并打印它的 href

## 来源与延伸

- CSS 选择器（MDN 中文，最权威）：https://developer.mozilla.org/zh-CN/docs/Learn/CSS/Building_blocks/Selectors
- BeautifulSoup 的 select 用法：https://www.crummy.com/software/BeautifulSoup/bs4/doc/#css-selectors
- 下一课：XPath——另一种 equally 强大的定位语言，Scrapy 默认用它。